# CERDOS 27 — Frozen-threshold pig counting evaluation
**Runtime: CPU (GPU NOT required)**  
**First manuscript — FaroPigSeg benchmark.** Descriptive evaluation on previously inspected **TEST** images. No model selection or parameter tuning using test. Thresholds are loaded **only** from experiment 26 validation provenance.  
**Figures:** English only; PNG 600 dpi, PDF, SVG. **Tabular results:** Excel. **One notebook:** `27_CERDOS_Evaluacion_Conteo_Calibrado_CPU.ipynb`.

In [ ]:
!pip -q install pandas numpy matplotlib xlsxwriter
from pathlib import Path
import io, json, zipfile, hashlib, datetime
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from google.colab import drive, files
from IPython.display import display

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Project not found: {ROOT}'
OUT=ROOT/'03_Resultados'/'CERDOS_27_Evaluacion_Conteo_Calibrado'
FIG=OUT/'Figures'; FIG.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.labelsize':12,
 'axes.spines.top':False,'axes.spines.right':False,'savefig.dpi':600})
SEEDS=(2026,2027,2028)
print('CPU notebook. Output:',OUT)

## 1. Load the archived model predictions and frozen validation selection
This notebook will not search for weights or rerun inference. When a ZIP is absent from Drive you may upload it. All thresholds and hashes are read from experiment 26; none is estimated on TEST.

In [ ]:
def get_zip(name):
    hits=sorted(ROOT.rglob(name),key=lambda p:len(p.parts))
    if len(hits)>1: print('Multiple files found; using:',hits[0])
    if hits:return hits[0]
    print('Upload:',name)
    upl=files.upload()
    if name not in upl:raise FileNotFoundError(f'Expected {name}, uploaded {list(upl)}')
    return Path('/content')/name

p23=get_zip('CERDOS_23_YOLO26_Evaluacion_Resultados.zip')
p26=get_zip('CERDOS_26_Resultados.zip')
with zipfile.ZipFile(p26) as z:
    prov26=json.loads(z.read('CERDOS_26_Provenance.json'))
with zipfile.ZipFile(p23) as z:
    audit=pd.read_excel(io.BytesIO(z.read('CERDOS_23_YOLO26_Evaluacion_COCO_3_Semillas.xlsx')),
                        sheet_name='Conteo_Por_Imagen')
    protocol=json.loads(z.read('CERDOS_23_Protocolo.json'))
    predictions={s:json.loads(z.read(f'CERDOS_23_Pred_seed_{s}.json')) for s in SEEDS}

selected=pd.DataFrame(prov26['selected'])
assert set(selected['seed'].astype(int))==set(SEEDS) and len(selected)==3, 'Invalid validation selection'
THRESHOLDS=dict(zip(selected['seed'].astype(int),selected['threshold'].astype(float)))
assert all(0.001<=t<=1 for t in THRESHOLDS.values())
assert prov26['validation_images']==196, 'Unexpected validation cohort'
assert set(audit.semilla.astype(int))==set(SEEDS)
assert (audit.groupby('semilla').image_id.nunique()==373).all()
assert not audit.duplicated(['semilla','image_id']).any()
assert all(set(['score','image_id']).issubset(d.keys()) for v in predictions.values() for d in v)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''):h.update(c)
    return h.hexdigest()
actual23=sha256_file(p23)
assert actual23==prov26['source_experiment23_sha256'], 'EXPERIMENT 23 ARCHIVE MISMATCH — stop'
assert set(audit.fecha.astype(str)).isdisjoint({'2023-06-06','2023-06-08','2023-09-01','2023-09-04'}), 'Validation dates present in TEST'
assert set(audit.fecha.astype(str))=={'2023-05-24','2023-08-21','2023-08-25','2023-08-29','2023-08-31'}
assert prov26['inference']=={'imgsz':640,'conf':0.001,'iou':0.7,'max_det':100,'retina_masks':True}, 'Inference protocol mismatch'
print('SOURCE VERIFIED: identical EXP23 ZIP hash, expected TEST dates, frozen validation thresholds.')
print('Frozen thresholds:',THRESHOLDS)
display(selected)

## 2. Count per image under the prespecified baseline and frozen thresholds
Counting is the number of archived COCO predictions at confidence score ≥ threshold. This **does not modify NMS or re-run the model**. A consistency assertion reproduces the original 0.25 count exactly.

In [ ]:
baseline=audit[['semilla','image_id','fecha','imagen','gt_conteo','pred_conteo_025']].copy()
blocks=[]
for seed in SEEDS:
    part=baseline.loc[baseline.semilla==seed].copy()
    det=pd.DataFrame([(int(x['image_id']),float(x['score'])) for x in predictions[seed]],columns=['image_id','score'])
    assert det.score.between(0,1).all()
    for name,t in [('Original (0.25)',0.25),('Validation-calibrated',THRESHOLDS[seed])]:
        counts=det.loc[det.score>=t].groupby('image_id').size()
        d=part.copy()
        d['strategy']=name
        d['threshold']=t
        d['predicted_count']=d.image_id.map(counts).fillna(0).astype(int)
        d['signed_error']=d.predicted_count-d.gt_conteo
        d['absolute_error']=d.signed_error.abs()
        blocks.append(d)
per_image=pd.concat(blocks,ignore_index=True)
original=per_image.query('strategy == "Original (0.25)"')
assert (original.predicted_count.values==original.pred_conteo_025.values).all(), 'Baseline mismatch!'
assert (per_image.groupby(['semilla','strategy']).size()==373).all()
print('Baseline reproduction PASSED. Per-image records:',len(per_image))

In [ ]:
def summarise(g):
    e=g.signed_error.to_numpy()
    return pd.Series({'n_images':len(g),'MAE':np.abs(e).mean(),
                      'RMSE':np.sqrt(np.mean(e**2)),'Bias':e.mean(),
                      'Exact_count_pct':100*np.mean(e==0),
                      'Overcount_pct':100*np.mean(e>0),
                      'Undercount_pct':100*np.mean(e<0)})
global_summary=per_image.groupby(['semilla','strategy'],sort=True).apply(summarise).reset_index()
by_date=per_image.groupby(['semilla','strategy','fecha'],sort=True).apply(summarise).reset_index()
w=per_image.pivot(index=['semilla','image_id','fecha','imagen','gt_conteo'],columns='strategy',values='absolute_error').reset_index()
w['MAE_difference_calibrated_minus_original']=w['Validation-calibrated']-w['Original (0.25)']
paired=w.groupby('semilla')['MAE_difference_calibrated_minus_original'].agg(['mean','median','std','count']).reset_index()
print('All results are DESCRIPTIVE on a previously inspected TEST dataset.')
display(global_summary,paired)

## 3. Publication-ready figures (English)
The two strategies are prespecified for this report. All test findings are descriptive because the held-out cohort has been inspected in preceding experiments.

In [ ]:
def save_fig(fig,name):
    fig.tight_layout()
    for ext in ('png','pdf','svg'):
        fig.savefig(FIG/f'{name}.{ext}',dpi=600,bbox_inches='tight')
    plt.show();plt.close(fig)

palette=['#0072B2','#D55E00','#009E73']
fig,ax=plt.subplots(figsize=(7.6,4.6))
x=np.arange(3);width=.34
for i,(st,color) in enumerate([('Original (0.25)','#6B7C85'),('Validation-calibrated','#0072B2')]):
    v=[float(global_summary.query('semilla == @s and strategy == @st').MAE.iloc[0]) for s in SEEDS]
    ax.bar(x+(i-.5)*width,v,width,color=color,label=st)
ax.set_xticks(x,[f'Seed {s}' for s in SEEDS]);ax.set_ylabel('Mean absolute counting error (pigs/image)')
ax.set_title('Counting error with frozen validation thresholds')
ax.legend(frameon=False);ax.grid(axis='y',alpha=.2)
save_fig(fig,'Fig27_01_Test_Count_MAE_Frozen')

fig,ax=plt.subplots(figsize=(8.4,4.8))
for s,color in zip(SEEDS,palette):
    d=by_date.query('semilla == @s and strategy == "Validation-calibrated"').sort_values('fecha')
    ax.plot(d.fecha.str[5:],d.MAE,marker='o',label=f'Seed {s}',color=color,linewidth=2)
ax.set(xlabel='Acquisition date (2023)',ylabel='Mean absolute counting error (pigs/image)',
       title='Calibrated counting error across acquisition dates')
ax.legend(frameon=False);ax.grid(alpha=.2)
save_fig(fig,'Fig27_02_Test_Calibrated_MAE_by_Date')

fig,ax=plt.subplots(figsize=(7.4,4.8))
for s,color in zip(SEEDS,palette):
    d=global_summary.query('semilla == @s')
    ax.plot([0,1],[float(d.loc[d.strategy=='Original (0.25)','Bias'].iloc[0]),
                      float(d.loc[d.strategy=='Validation-calibrated','Bias'].iloc[0])],
            marker='o',linewidth=2,color=color,label=f'Seed {s}')
ax.axhline(0,ls='--',color='black',lw=1)
ax.set_xticks([0,1],['Original (0.25)','Validation-calibrated'])
ax.set(ylabel='Mean signed counting error (pigs/image)',title='Counting bias before and after calibration')
ax.legend(frameon=False);ax.grid(axis='y',alpha=.2)
save_fig(fig,'Fig27_03_Test_Count_Bias_Frozen')

## 4. Export tables, provenance and ZIP to Drive
**Important:** experiment 27 measures results on a *previously inspected* test cohort, not a new untouched or external test set. No threshold optimization takes place here.

In [ ]:
excel=OUT/'CERDOS_27_Frozen_Threshold_Test_Evaluation.xlsx'
with pd.ExcelWriter(excel,engine='xlsxwriter') as writer:
    tables={'Global_Comparison':global_summary,'By_Date':by_date,
            'Per_Image':per_image,'Paired_Image_Deltas':w,'Paired_Summary':paired,
            'Locked_Thresholds':selected}
    for tab,d in tables.items():
        d.to_excel(writer,sheet_name=tab,index=False)
        ws=writer.sheets[tab]
        ws.freeze_panes(1,0);ws.autofilter(0,0,len(d),len(d.columns)-1)
        for j,col in enumerate(d.columns):ws.set_column(j,j,min(42,max(16,len(col)+3)))
        head=writer.book.add_format({'bold':True,'font_color':'white','bg_color':'#17365D'})
        for j,col in enumerate(d.columns):ws.write(0,j,col,head)
meta={'experiment':27,'runtime':'CPU','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'analysis':'DESCRIPTIVE TEST - previous test inspection; not external validation',
      'experiment23_zip':str(p23),'experiment23_sha256':actual23,
      'experiment26_zip':str(p26),'experiment26_sha256':sha256_file(p26),
      'locked_thresholds':THRESHOLDS,'prediction_rule':'Count COCO detections with score >= threshold',
      'nms_note':'Archived inference NMS remains fixed; no model rerun',
      'warning':'Test previously inspected during experiments 23-25; do not tune on test or claim an untouched holdout.'}
(OUT/'CERDOS_27_Provenance.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
archive=OUT/'CERDOS_27_Resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in [excel,OUT/'CERDOS_27_Provenance.json',*sorted(FIG.glob('*'))]:z.write(f,f.relative_to(OUT))
print('COMPLETED:',archive)
files.download(str(archive))